# Neural Forest — Colab Training

Trains **Neural Forest tiny** (~47.8M params) on **100M tokens** of TinyStories.

| Item | Value |
|------|-------|
| Model | ForestConfig.tiny() — 47.8M params |
| Dataset | TinyStories (100M tokens, GPT-2 BPE) |
| Steps | 5 000 |
| Batch | 32 × 512 tokens = 16 384 tok/step |
| Expected time | ~35 min on A100 |
| Expected loss | 10.8 → ~3.5 |

**Before running:** Runtime → Change runtime type → **A100 GPU**

In [ ]:
# Cell 1 — Verify GPU
!nvidia-smi
import torch
print(f"\nPyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {vram:.1f} GB")

In [ ]:
# Cell 2 — Clone repo and install dependencies
!git clone https://github.com/naoufelbvb2-arch/neural-forest-.git neural-forest
%cd neural-forest

!pip install -q -e .
!pip install -q wandb datasets tiktoken tqdm accelerate

print("\nInstallation complete.")

In [ ]:
# Cell 3 — Login to Weights & Biases
# Get your API key from: https://wandb.ai/authorize
import wandb
wandb.login()

In [ ]:
# Cell 4 — Run training
# W&B dashboard link will appear after the first step.
!python scripts/train_colab.py

In [ ]:
# Cell 5 — Download final checkpoint
from google.colab import files
files.download("checkpoints/tiny_100m_final.pt")

In [ ]:
# Cell 6 — Quick sanity check: load checkpoint and run one forward pass
import torch
from forest.config import ForestConfig
from forest.core.model import NeuralForest

ckpt = torch.load("checkpoints/tiny_100m_final.pt", map_location="cpu")
config = ckpt["config"]
model  = NeuralForest(config)
model.load_state_dict(ckpt["model_state"])
model.eval()

dummy = torch.randint(0, config.vocab_size, (1, 16))
with torch.no_grad():
    out = model(dummy)

print(f"Checkpoint step  : {ckpt['step']}")
print(f"Logits shape     : {out['logits'].shape}")
print(f"Zones used       : {out['routing_decision'].zone_indices.unique().tolist()}")
print("Checkpoint OK.")